# Active Set QP Methods for Single-Factor Covariance Model
**Author:** Jonathan Cushman  
**Date:** October 2026  

## Libraries

In [25]:
import numpy as np

## Feasible starting Portfolio

In [26]:
def feasible_start(mu, target, tol=1e-10):
    p = len(mu)
    best_w = None
    best_min_weight = -np.inf

    for i in range(p):
        for j in range(i + 1, p):
            if abs(mu[j] - mu[i]) < tol:
                continue

            wi = (mu[j] - target) / (mu[j] - mu[i])
            wj = (target - mu[i]) / (mu[j] - mu[i])

            if wi >= -tol and wj >= -tol:
                w = np.zeros(p)
                w[i] = wi
                w[j] = wj
                w[np.abs(w) < tol] = 0.0

                if min(w[i], w[j]) > best_min_weight:
                    best_w = w
                    best_min_weight = min(w[i], w[j])

    if best_w is None:
        raise ValueError(
            "Could not construct a feasible starting portfolio."
        )

    return best_w

## Active Set Algorithm

In [27]:
def active_set_qp(V, mu, target, tol=1e-9, max_iter=500):

    V = np.asarray(V, dtype=float)
    mu = np.asarray(mu, dtype=float)
    p = len(mu)

    if V.shape != (p, p):
        raise ValueError("V must be a square matrix matching mu.")

    if not np.allclose(V, V.T, atol=tol):
        raise ValueError("V must be symmetric.")

    if np.min(np.linalg.eigvalsh(V)) <= 0:
        raise ValueError("V must be positive definite.")

    if target < np.min(mu) - tol or target > np.max(mu) + tol:
        raise ValueError("Target return is outside the feasible range.")

    E = np.column_stack((np.ones(p), mu))
    b = np.array([1.0, target])

    w = feasible_start(mu, target, tol)

    active = set(np.flatnonzero(w <= tol))

    for iteration in range(max_iter):
        free = np.array([i for i in range(p) if i not in active], dtype=int)
        active_idx = np.array(sorted(active), dtype=int)

        if len(free) < 2 or np.linalg.matrix_rank(E[free].T) < 2:
            print("Iteration:", iteration)
            print("Weights:", w)
            print("Active set:", active)
            print("Free set:", free)
            print("Rank:", np.linalg.matrix_rank(E[free].T))
            raise RuntimeError("Degenerate working set: equality constraints"
                     "are not independent on the free assets.")

        H = 2.0 * V[np.ix_(free, free)]
        E_free = E[free, :]
        grad_free = 2.0 * (V @ w)[free]

        KKT = np.block([
            [H, -E_free],
            [E_free.T, np.zeros((2, 2))]
        ])

        rhs = np.concatenate((-grad_free, np.zeros(2)))
        solution = np.linalg.solve(KKT, rhs)

        direction_free = solution[:len(free)]
        direction = np.zeros(p)
        direction[free] = direction_free

        if np.linalg.norm(direction, ord=np.inf) <= tol:
            grad = 2.0 * V @ w
            eta, *_ = np.linalg.lstsq(E_free, grad_free, rcond=None)

            nu_active = grad[active_idx] - E[active_idx] @ eta

            if len(active_idx) == 0 or np.min(nu_active) >= -tol:
                return {
                    "weights": w,
                    "variance": float(w @ V @ w),
                    "iterations": iteration + 1,
                    "active": sorted(active),
                    "lambda": float(eta[0]),
                    "gamma": float(eta[1]),
                    "multipliers": dict(
                        zip(active_idx.tolist(), nu_active.tolist())
                    ),
                }

            remove_pos = int(np.argmin(nu_active))
            active.remove(int(active_idx[remove_pos]))
            continue

        alpha = 1.0
        blocker = None

        for i in free:
            if direction[i] < -tol:
                candidate = -w[i] / direction[i]

                if candidate < alpha:
                    alpha = candidate
                    blocker = int(i)

        w = w + alpha * direction

        w[np.abs(w) < tol] = 0.0

        if np.min(w) < -10 * tol:
            raise RuntimeError("Numerical loss of feasibility.")

        w = np.maximum(w, 0.0)

        if not np.allclose(E.T @ w, b, atol=100 * tol):
            raise RuntimeError("Equality constraints lost feasibility.")

        if blocker is not None and alpha < 1.0 - tol:
            active.add(blocker)

    raise RuntimeError("Maximum number of iterations reached.")

## Basic Testing 

In [28]:

mu = np.array([0.05, 0.10, 0.15])
V = np.eye(3)
target = 0.10

result = active_set_qp(V, mu, target)

print("Optimal weights:", result["weights"])
print("Portfolio variance:", result["variance"])
print("Iterations:", result["iterations"])
print("Active assets (zero-based indices):", result["active"])
print("Equality multipliers:")
print("lambda =", result["lambda"])
print("gamma  =", result["gamma"])
print("Active multipliers:", result["multipliers"])

Optimal weights: [0.33333333 0.33333333 0.33333333]
Portfolio variance: 0.3333333333333333
Iterations: 3
Active assets (zero-based indices): []
Equality multipliers:
lambda = 0.6666666666666666
gamma  = -1.3714729665606958e-17
Active multipliers: {}
